In [3]:
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")


Đang sử dụng thiết bị: cpu


In [4]:
DATA_PATH = r"D:\2026\NLP\IMDB Dataset.csv"
df = pd.read_csv(DATA_PATH)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})


def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    return text.split()


print("Đang xử lý text...")
tokens_list = [clean_text(review) for review in df['review']]

VOCAB_SIZE = 25000
word_counts = Counter(word for doc in tokens_list for word in doc)
vocab = {
    word: i + 2
    for i, (word, _) in enumerate(word_counts.most_common(VOCAB_SIZE))
}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Tổng số từ trong từ điển: {len(vocab)}")


Đang xử lý text...
Tổng số từ trong từ điển: 25002


In [8]:
import gensim.downloader as api
import numpy as np
import torch

print("Đang tải pre-trained FastText từ thư viện gensim...")
fasttext_model = api.load("fasttext-wiki-news-subwords-300")
print("Tải FastText thành công!")

EMBED_DIM = 300

weights_matrix = np.zeros((len(vocab), EMBED_DIM))
words_found = 0

for word, idx in vocab.items():
    if word in fasttext_model:
        weights_matrix[idx] = fasttext_model[word]
        words_found += 1
    else:
        weights_matrix[idx] = np.random.normal(scale=0.6, size=(EMBED_DIM, ))

weights_matrix[0] = np.zeros(EMBED_DIM)
weights_matrix = torch.tensor(weights_matrix, dtype=torch.float32)

print(f"Khớp thành công {words_found}/{len(vocab)} từ với FastText!")


Đang tải pre-trained FastText từ thư viện gensim...
[==================================================] 100.0% 958.5/958.4MB downloaded
Tải FastText thành công!
Khớp thành công 22988/25002 từ với FastText!


In [9]:
MAX_LEN = 150


def encode_pad(tokens, max_len=MAX_LEN):
    indices = [vocab.get(w, 1) for w in tokens]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices


X = [encode_pad(tokens) for tokens in tokens_list]
y = df['sentiment'].values


class IMDBDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


SPLIT = 40000
train_dataset = IMDBDataset(X[:SPLIT], y[:SPLIT])
test_dataset = IMDBDataset(X[SPLIT:], y[SPLIT:])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [10]:
class First_RNN(nn.Module):

    def __init__(self,
                 embedding_weights,
                 hidden_dim,
                 output_dim=1,
                 freeze_embed=False):
        super(First_RNN, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]

        self.i2h = nn.Linear(embed_dim, hidden_dim)
        self.h2h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.tanh = nn.Tanh()

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        batch_size, seq_len = text.size()
        embeds = self.embedding(text)

        h = torch.zeros(batch_size, self.hidden_dim).to(text.device)

        for t in range(seq_len):
            x_t = embeds[:, t, :]
            h = self.tanh(self.i2h(x_t) + self.h2h(h))

        return self.fc(h).squeeze(1)


In [ ]:
class Second_RNN(nn.Module):
    """RNN nhieu tang, 2 chieu + max/mean pooling.

    Cai tien so voi ban cu:
      - Bidirectional: hoc ngu canh ca 2 chieu.
      - Max + Mean pooling tren toan bo timestep thay vi chi hidden cuoi
        -> giam anh huong cua vanishing gradient tren chuoi dai.
      - Dropout o embedding, giua cac tang RNN va truoc FC.
      - LayerNorm + khoi tao Xavier cho FC.
    """

    def __init__(self,
                 embedding_weights,
                 hidden_dim=128,
                 num_layers=2,
                 output_dim=1,
                 dropout=0.3,
                 freeze_embed=False,
                 bidirectional=True):
        super(Second_RNN, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]
        self.embed_dropout = nn.Dropout(dropout)

        self.rnn = nn.RNN(embed_dim,
                          hidden_dim,
                          num_layers=num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0,
                          bidirectional=bidirectional)

        num_directions = 2 if bidirectional else 1
        rnn_out_dim = hidden_dim * num_directions

        self.layer_norm = nn.LayerNorm(rnn_out_dim * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(rnn_out_dim * 2, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, text):
        embeds = self.embed_dropout(self.embedding(text))
        out, _ = self.rnn(embeds)

        mask = (text != 0).unsqueeze(-1)

        out_masked = out.masked_fill(~mask, -1e9)
        max_pool = out_masked.max(dim=1).values

        lengths = mask.sum(dim=1).clamp(min=1)
        mean_pool = (out * mask).sum(dim=1) / lengths

        pooled = torch.cat([max_pool, mean_pool], dim=1)
        pooled = self.layer_norm(pooled)
        pooled = self.fc_dropout(pooled)

        return self.fc(pooled).squeeze(1)


In [ ]:
class First_LSTM(nn.Module):

    def __init__(self,
                 embedding_weights,
                 hidden_dim,
                 output_dim=1,
                 freeze_embed=False):
        super(First_LSTM, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]

        self.lstm = nn.LSTM(embed_dim,
                            hidden_dim,
                            num_layers=1,
                            batch_first=True)

        self.fc = nn.Linear(hidden_dim, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, text):
        embeds = self.embedding(text)
        out, (h_n, c_n) = self.lstm(embeds)

        return self.fc(h_n[-1]).squeeze(1)


In [ ]:
class Second_LSTM(nn.Module):
    """LSTM nhiều tầng, 2 chiều + max/mean pooling.

    Tương tự Second_RNN nhưng dùng LSTM để giảm vanishing gradient.
    """

    def __init__(self,
                 embedding_weights,
                 hidden_dim=128,
                 num_layers=2,
                 output_dim=1,
                 dropout=0.3,
                 freeze_embed=False,
                 bidirectional=True):
        super(Second_LSTM, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]
        self.embed_dropout = nn.Dropout(dropout)

        self.rnn = nn.LSTM(embed_dim,
                           hidden_dim,
                           num_layers=num_layers,
                           batch_first=True,
                           dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)

        num_directions = 2 if bidirectional else 1
        rnn_out_dim = hidden_dim * num_directions

        self.layer_norm = nn.LayerNorm(rnn_out_dim * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(rnn_out_dim * 2, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, text):
        embeds = self.embed_dropout(self.embedding(text))
        out, _ = self.rnn(embeds)

        mask = (text != 0).unsqueeze(-1)

        out_masked = out.masked_fill(~mask, -1e9)
        max_pool = out_masked.max(dim=1).values

        lengths = mask.sum(dim=1).clamp(min=1)
        mean_pool = (out * mask).sum(dim=1) / lengths

        pooled = torch.cat([max_pool, mean_pool], dim=1)
        pooled = self.layer_norm(pooled)
        pooled = self.fc_dropout(pooled)

        return self.fc(pooled).squeeze(1)


In [ ]:
class First_GRU(nn.Module):
    """GRU tu code thu cong (manual GRU cell)."""

    def __init__(self,
                 embedding_weights,
                 hidden_dim,
                 output_dim=1,
                 freeze_embed=False):
        super(First_GRU, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]

        self.x2h = nn.Linear(embed_dim, 3 * hidden_dim)
        self.h2h = nn.Linear(hidden_dim, 3 * hidden_dim)

        self.fc = nn.Linear(hidden_dim, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, text):
        batch_size, seq_len = text.size()
        embeds = self.embedding(text)

        h = torch.zeros(batch_size, self.hidden_dim, device=text.device)

        for t in range(seq_len):
            x_t = embeds[:, t, :]
            x_r, x_z, x_n = self.x2h(x_t).chunk(3, dim=1)
            h_r, h_z, h_n = self.h2h(h).chunk(3, dim=1)

            r = torch.sigmoid(x_r + h_r)
            z = torch.sigmoid(x_z + h_z)
            n = torch.tanh(x_n + r * h_n)
            h = (1 - z) * n + z * h

        return self.fc(h).squeeze(1)


In [ ]:
class Second_GRU(nn.Module):
    """GRU nhieu tang, 2 chieu + max/mean pooling (dung nn.GRU)."""

    def __init__(self,
                 embedding_weights,
                 hidden_dim=128,
                 num_layers=2,
                 output_dim=1,
                 dropout=0.3,
                 freeze_embed=False,
                 bidirectional=True):
        super(Second_GRU, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=freeze_embed, padding_idx=0)
        embed_dim = embedding_weights.shape[1]
        self.embed_dropout = nn.Dropout(dropout)

        self.rnn = nn.GRU(embed_dim,
                          hidden_dim,
                          num_layers=num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0,
                          bidirectional=bidirectional)

        num_directions = 2 if bidirectional else 1
        rnn_out_dim = hidden_dim * num_directions

        self.layer_norm = nn.LayerNorm(rnn_out_dim * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(rnn_out_dim * 2, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, text):
        embeds = self.embed_dropout(self.embedding(text))
        out, _ = self.rnn(embeds)

        mask = (text != 0).unsqueeze(-1)

        out_masked = out.masked_fill(~mask, -1e9)
        max_pool = out_masked.max(dim=1).values

        lengths = mask.sum(dim=1).clamp(min=1)
        mean_pool = (out * mask).sum(dim=1) / lengths

        pooled = torch.cat([max_pool, mean_pool], dim=1)
        pooled = self.layer_norm(pooled)
        pooled = self.fc_dropout(pooled)

        return self.fc(pooled).squeeze(1)


In [ ]:
"""Shared training routine for First_RNN, Second_RNN, First_LSTM, Second_LSTM."""

import copy

import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score

__all__ = [
    "EarlyStopping",
    "compute_metrics",
    "train_epoch",
    "evaluate",
    "fit",
    "train_model",
]


def _model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")


def compute_metrics(y_true, y_prob, loss):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = (np.asarray(y_prob).ravel() >= 0.5).astype(int)
    return {
        "loss": float(loss),
        "acc": float((y_pred == y_true).mean()),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


class EarlyStopping:
    """Stop training when the validation metric stops improving."""

    def __init__(self, patience=3, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None

    def __call__(self, score, model):
        improved = self.best_score is None
        if not improved:
            if self.mode == "max":
                improved = score > self.best_score + self.min_delta
            else:
                improved = score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    """Train for one epoch and return loss/acc/precision/recall/f1."""
    device = _model_device(model)
    model.train()
    total_loss = 0.0
    probs, targets = [], []

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        if clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
        optimizer.step()

        total_loss += loss.item()
        probs.append(torch.sigmoid(logits).detach().cpu())
        targets.append(y_batch.detach().cpu())

    return compute_metrics(
        torch.cat(targets).numpy(),
        torch.cat(probs).numpy(),
        total_loss / len(loader),
    )


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate the model and return loss/acc/precision/recall/f1."""
    device = _model_device(model)
    model.eval()
    total_loss = 0.0
    probs, targets = [], []

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item()
        probs.append(torch.sigmoid(logits).cpu())
        targets.append(y_batch.cpu())

    return compute_metrics(
        torch.cat(targets).numpy(),
        torch.cat(probs).numpy(),
        total_loss / len(loader),
    )


def fit(model, train_loader, val_loader, optimizer, criterion, epochs,
        scheduler=None, early_stopping=None, model_name="model"):
    """Full training loop with metrics logging, scheduler and early stopping."""
    history = {k: [] for k in [
        "epoch", "lr", "train_loss", "train_acc", "train_f1",
        "val_loss", "val_acc", "val_precision", "val_recall", "val_f1",
    ]}

    for epoch in range(1, epochs + 1):
        lr = optimizer.param_groups[0]["lr"]
        tr = train_epoch(model, train_loader, optimizer, criterion)
        va = evaluate(model, val_loader, criterion)

        if scheduler is not None:
            scheduler.step(va["f1"])

        history["epoch"].append(epoch)
        history["lr"].append(lr)
        for k in ["loss", "acc", "f1"]:
            history["train_" + k].append(tr[k])
        for k in ["loss", "acc", "precision", "recall", "f1"]:
            history["val_" + k].append(va[k])

        print(f"[{model_name}] Epoch {epoch:02d}/{epochs} | lr {lr:.2e}")
        print(f"   Train | loss {tr['loss']:.4f} | acc {tr['acc'] * 100:.2f}% "
              f"| f1 {tr['f1'] * 100:.2f}%")
        print(f"   Val   | loss {va['loss']:.4f} | acc {va['acc'] * 100:.2f}% "
              f"| P {va['precision'] * 100:.2f}% | R {va['recall'] * 100:.2f}% "
              f"| F1 {va['f1'] * 100:.2f}%")

        if early_stopping is not None and early_stopping(va["f1"], model):
            print(f"[{model_name}] >> Early stopping o epoch {epoch} "
                  f"(best val F1 = {early_stopping.best_score * 100:.2f}%)")
            break

    if early_stopping is not None and early_stopping.best_state is not None:
        model.load_state_dict(early_stopping.best_state)
        print(f"[{model_name}] >> Khoi phuc trong so tot nhat "
              f"(val F1 = {early_stopping.best_score * 100:.2f}%)")

    return history


def train_model(model, train_loader, val_loader, test_loader, criterion,
                epochs=10, lr=1e-3, weight_decay=0.0,
                optimizer_cls=torch.optim.AdamW,
                scheduler_patience=1, early_stopping_patience=3,
                min_delta=1e-4, clip=1.0, model_name="model"):
    """The single shared training routine used by all four models.

    Returns (history, test_metrics). test_metrics is None when test_loader
    is None.
    """
    optimizer = optimizer_cls(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=scheduler_patience
    )
    early_stopping = EarlyStopping(
        patience=early_stopping_patience, min_delta=min_delta, mode="max"
    )

    print("=" * 20 + f" TRAIN {model_name} " + "=" * 20)
    history = fit(
        model, train_loader, val_loader, optimizer, criterion,
        epochs=epochs, scheduler=scheduler,
        early_stopping=early_stopping, model_name=model_name,
    )

    test_metrics = None
    if test_loader is not None:
        test_metrics = evaluate(model, test_loader, criterion)
        print("\n" + "=" * 20 + f" TEST {model_name} " + "=" * 20)
        print(f"Loss     : {test_metrics['loss']:.4f}")
        print(f"Accuracy : {test_metrics['acc'] * 100:.2f}%")
        print(f"Precision: {test_metrics['precision'] * 100:.2f}%")
        print(f"Recall   : {test_metrics['recall'] * 100:.2f}%")
        print(f"F1-score : {test_metrics['f1'] * 100:.2f}%")

    return history, test_metrics


In [ ]:
from torch.utils.data import random_split, DataLoader

EPOCHS = 15
HIDDEN_DIM = 64
HIDDEN_DIM_2 = 128
BATCH_SIZE = 64
VAL_RATIO = 0.1
criterion = nn.BCEWithLogitsLoss()

val_size = int(len(train_dataset) * VAL_RATIO)
train_size = len(train_dataset) - val_size
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size], generator=generator)

train_loader_v2 = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader_v2 = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader_v2 = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {train_size} | Val: {val_size} | Test: {len(test_dataset)}")


In [ ]:
model_1 = First_RNN(embedding_weights=weights_matrix,
                    hidden_dim=HIDDEN_DIM,
                    freeze_embed=True).to(device)

history_1, test_metrics_1 = train_model(
    model_1, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=0.0,
    optimizer_cls=torch.optim.Adam,
    model_name='First_RNN')


In [ ]:
model_2 = Second_RNN(embedding_weights=weights_matrix,
                     hidden_dim=HIDDEN_DIM_2,
                     num_layers=2,
                     dropout=0.3,
                     freeze_embed=True,
                     bidirectional=True).to(device)

history_2, test_metrics_2 = train_model(
    model_2, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=1e-4,
    optimizer_cls=torch.optim.AdamW,
    model_name='Second_RNN')


In [ ]:
model_3 = First_LSTM(embedding_weights=weights_matrix,
                     hidden_dim=HIDDEN_DIM,
                     freeze_embed=True).to(device)

history_3, test_metrics_3 = train_model(
    model_3, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=0.0,
    optimizer_cls=torch.optim.Adam,
    model_name='First_LSTM')


In [ ]:
model_4 = Second_LSTM(embedding_weights=weights_matrix,
                      hidden_dim=HIDDEN_DIM_2,
                      num_layers=2,
                      dropout=0.3,
                      freeze_embed=True,
                      bidirectional=True).to(device)

history_4, test_metrics_4 = train_model(
    model_4, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=1e-4,
    optimizer_cls=torch.optim.AdamW,
    model_name='Second_LSTM')


In [ ]:
model_5 = First_GRU(embedding_weights=weights_matrix,
                    hidden_dim=HIDDEN_DIM,
                    freeze_embed=True).to(device)

history_5, test_metrics_5 = train_model(
    model_5, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=0.0,
    optimizer_cls=torch.optim.Adam,
    model_name='First_GRU')


In [ ]:
model_6 = Second_GRU(embedding_weights=weights_matrix,
                     hidden_dim=HIDDEN_DIM_2,
                     num_layers=2,
                     dropout=0.3,
                     freeze_embed=True,
                     bidirectional=True).to(device)

history_6, test_metrics_6 = train_model(
    model_6, train_loader_v2, val_loader_v2, test_loader_v2, criterion,
    epochs=EPOCHS, lr=1e-3, weight_decay=1e-4,
    optimizer_cls=torch.optim.AdamW,
    model_name='Second_GRU')


In [ ]:
import copy

from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

weights_np = weights_matrix.cpu().numpy()


def mean_pool(indices_list):
    feats = np.zeros((len(indices_list), weights_np.shape[1]), dtype=np.float32)
    for i, ids in enumerate(indices_list):
        ids = np.asarray(ids)
        keep = ids != 0
        if keep.any():
            feats[i] = weights_np[ids[keep]].mean(axis=0)
    return feats


def clf_metrics(y_true, y_pred):
    return {
        'loss': np.nan,
        'acc': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }


train_idx = np.asarray(train_subset.indices)
val_idx = np.asarray(val_subset.indices)

X_train_pool = mean_pool([X[i] for i in train_idx])
X_val_pool = mean_pool([X[i] for i in val_idx])
X_test_pool = mean_pool(X[SPLIT:])
y_train, y_val, y_test = y[train_idx], y[val_idx], y[SPLIT:]

scaler = StandardScaler()
X_train_pool = scaler.fit_transform(X_train_pool)
X_val_pool = scaler.transform(X_val_pool)
X_test_pool = scaler.transform(X_test_pool)

best_c, best_f1, svm = None, -1.0, None
for c in [0.01, 0.1, 1.0]:
    candidate = LinearSVC(C=c, max_iter=5000, random_state=42)
    candidate.fit(X_train_pool, y_train)
    val_f1 = f1_score(y_val, candidate.predict(X_val_pool), zero_division=0)
    print(f'[SVM] C={c:<5} | val F1={val_f1 * 100:.2f}%')
    if val_f1 > best_f1:
        best_c, best_f1, svm = c, val_f1, candidate

svm_test = svm.predict(X_test_pool)
print(f'[SVM] best C={best_c} | val F1={best_f1 * 100:.2f}%')
print(f'[SVM] test | acc {accuracy_score(y_test, svm_test) * 100:.2f}% '
      f'| F1 {f1_score(y_test, svm_test) * 100:.2f}%')
test_metrics_svm = clf_metrics(y_test, svm_test)

mlp = MLPClassifier(hidden_layer_sizes=(256,),
                    alpha=1e-3,
                    batch_size=256,
                    learning_rate_init=1e-3,
                    max_iter=1,
                    random_state=42)

best_f1, best_mlp, counter = -1.0, None, 0
for epoch in range(1, 31):
    mlp.partial_fit(X_train_pool, y_train, classes=np.array([0, 1]))
    val_f1 = f1_score(y_val, mlp.predict(X_val_pool), zero_division=0)
    if val_f1 > best_f1 + 1e-4:
        best_f1, best_mlp, counter = val_f1, copy.deepcopy(mlp), 0
    else:
        counter += 1
    if epoch % 5 == 0:
        print(f'[MLP] epoch {epoch:02d} | val F1={val_f1 * 100:.2f}%')
    if counter >= 3:
        break

mlp = best_mlp
mlp_test = mlp.predict(X_test_pool)
print(f'[MLP] best val F1={best_f1 * 100:.2f}%')
print(f'[MLP] test | acc {accuracy_score(y_test, mlp_test) * 100:.2f}% '
      f'| F1 {f1_score(y_test, mlp_test) * 100:.2f}%')
test_metrics_mlp = clf_metrics(y_test, mlp_test)


In [ ]:
import pandas as pd

results = pd.DataFrame({
    'First_RNN': test_metrics_1,
    'Second_RNN': test_metrics_2,
    'First_LSTM': test_metrics_3,
    'Second_LSTM': test_metrics_4,
    'First_GRU': test_metrics_5,
    'Second_GRU': test_metrics_6,
    'SVM (mean pool)': test_metrics_svm,
    'MLP (mean pool)': test_metrics_mlp,
}).T

results
